# 🎬 CineScope: Core Data & Model Training Pipeline
This notebook serves as the interactive testing ground and offline pre-computation pipeline for the CineScope Hybrid Recommendation Engine. It executes the extraction of TF-IDF metadata matrices, trains the SVD Collaborative Filtering parameters, and validates structural tracking databases.

### 📦 Step 1: Environment Diagnostics & Dependencies

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
import sqlite3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

print(f"Python Version: {sys.version}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")

Python Version: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Pandas Version: 3.0.2
NumPy Version: 1.26.4


### 📂 Step 2: Path Normalization & Data Extraction

In [4]:
# Set up workspace anchors relative to the execution root
ROOT_DIR = Path(os.getcwd())
DATA_DIR = Path("C:/Users/ZoroDM/cinescope-fresh/data/ml-latest-small")
MODEL_DIR = Path("C:/Users/ZoroDM/cinescope-fresh/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root Project Workspace: {ROOT_DIR}")
print(f"Data Directory Status: {DATA_DIR.exists()}")

Root Project Workspace: c:\Users\ZoroDM\cinescope-fresh\notebook
Data Directory Status: True


### 🧠 Step 3: Build Content-Based Filtering Matrices (TF-IDF)
This block reads your structured metadata files, processes genre string combinations, builds vocabulary maps, and exports the serialized matrix artifacts.

In [5]:
print("Parsing movies dataset metadata...")
movies_df = pd.read_csv(DATA_DIR / 'movies.csv')

# Clean up pipe-separated arrays into text documents
movies_df['genres_space'] = movies_df['genres'].str.replace('|', ' ', regex=False)
movies_df['genres_space'] = movies_df['genres_space'].fillna('')

print("Vectorizing genre matrices with TF-IDF...")
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(movies_df['genres_space'])

print(f"Matrix construction finalized. Structural Shape: {genre_matrix.shape}")

Parsing movies dataset metadata...
Vectorizing genre matrices with TF-IDF...
Matrix construction finalized. Structural Shape: (9742, 23)


### 🤖 Step 4: Train Collaborative Filtering Parameters (SVD)
This section maps explicit user behaviors out into low-dimensional latent spaces using Singular Value Decomposition.

In [6]:
print("Extracting collaborative feedback dimensions...")
ratings_df = pd.read_csv(DATA_DIR / 'ratings.csv')

# Parse into specialized dataset objects optimized for factorization extensions
reader = Reader(rating_scale=(0.5, 5.0))
data_stream = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating']], reader)

print("Optimizing SVD matrix factorization vectors...")
svd_engine = SVD(n_factors=100, random_state=42)
full_trainset = data_stream.build_full_trainset()
svd_engine.fit(full_trainset)

print("SVD parameter optimization complete.")

Extracting collaborative feedback dimensions...
Optimizing SVD matrix factorization vectors...
SVD parameter optimization complete.


### 📊 Step 5: Run Evaluation Metrics Suit (Precision, Recall, NDCG)
We cross-validate predictive strength on an un-trained test data split to evaluate actual performance thresholds.

In [7]:
from collections import defaultdict

# Split dataset into 80/20 train/test segments
train_split, test_split = train_test_split(data_stream, test_size=0.2, random_state=42)
eval_model = SVD(n_factors=100, random_state=42)
eval_model.fit(train_split)
test_predictions = eval_model.test(test_split)

def eval_pipeline(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    ndcgs = []
    
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) for (est, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0
        
        if len(user_ratings) >= 2:
            dcg = sum((2**true_r - 1) / np.log2(i + 2) for i, (_, true_r) in enumerate(user_ratings[:k]))
            user_ratings.sort(key=lambda x: x[1], reverse=True)
            idcg = sum((2**true_r - 1) / np.log2(i + 2) for i, (_, true_r) in enumerate(user_ratings[:k]))
            if idcg > 0:
                ndcgs.append(dcg / idcg)

    print(f"• Mean Precision@{k}: {np.mean(list(precisions.values())):.4f}")
    print(f"• Mean Recall@{k}:    {np.mean(list(recalls.values())):.4f}")
    print(f"• Mean NDCG@{k}:      {np.mean(ndcgs):.4f}")

eval_pipeline(test_predictions)

• Mean Precision@10: 0.7446
• Mean Recall@10:    0.5086
• Mean NDCG@10:      0.7949


### 💾 Step 6: Serialize Model Artifacts & Stage Telemetry

In [8]:
print("Exporting structural data binaries to storage volumes...")

with open(MODEL_DIR / 'movies.pkl', 'wb') as f:
    pickle.dump(movies_df, f)

with open(MODEL_DIR / 'genre_matrix.pkl', 'wb') as f:
    pickle.dump(genre_matrix, f)

with open(MODEL_DIR / 'svd_model.pkl', 'wb') as f:
    pickle.dump(svd_engine, f)

print("All binary engine artifacts saved successfully. Ready for deployment infrastructure pipelines.")

Exporting structural data binaries to storage volumes...
All binary engine artifacts saved successfully. Ready for deployment infrastructure pipelines.
